# An ILS deal analysis with prospicio

An investment fund gets an offer of a share of a cedant's 2026 premium as a **collateralised reinsurance** deal. The insurance-linked securities (ILS) market writes this kind of deal. The fund puts collateral in a trust, and the cedant pays premium into the trust. The trust pays the losses, and the fund gets the remainder when the deal commutes.

The fund asks about the return to its investors, not about the technical price. What return does the deal give, how likely is a loss, and how do the alternative structures compare? This notebook answers these questions in `prospicio`, in six steps:

1. read a submission and build the cedant's loss triangle,
2. show the historical exhibits and test how well a Bayesian model forecasts the cedant's history,
3. state an industry prior as an explicit input, and fit the cedant's ultimates against it,
4. draw the 2026 loss ratio, and apply the reinsurance structure, the collateral and the cash flows to it,
5. read the investor's IRR, MOIC and downside from the simulations, and compare two structures,
6. after binding, compare the next diagonal of actual losses with the one-year forecast.

The workflow is generic: a data submission, a loss model, a contract, a collateral rule and an investor.

**The data.** The submission is the synthetic one of `submission_evaluation.ipynb`. `examples/data/make_submission.py` wrote it from known parameters. Here the cedant offers a **20% quota share** of its commercial property book for 2026, not the per-risk layers of that notebook. Thus the loss to the fund is a share of the loss ratio of the cedant's whole book. All the deal terms below (the share, the commission, the collateral rule, the yield) are assumptions for the demonstration, not values in the files. Change them and run the notebook again.

To run the notebook, clone the repository and start Jupyter with prospicio built from source, which needs Rust. From the repository root, run `uv run --project python --group examples jupyter lab` (see `examples/README.md`). Set `PROSPICIO_EXAMPLE_SIMS` to run fewer simulations. The docs site publishes the results, and the repository keeps only the code.

In [ ]:
import csv
import math
import os
from statistics import NormalDist
from pathlib import Path

try:
    from prospicio.credibility import hierarchical_credibility
    from prospicio.distributions import Sampled
    from prospicio.pricing import (CashflowTerms, CollateralRequirement, DealBook, DealRun, Period, PaymentPattern,
                                   RateHistory, year_fraction)
    from prospicio.reinsurance import Layer, Tower
    from prospicio.reserving import BayesBfNormalised, BayesOdp, ChainLadder, Triangle, diagonal_backtest
    from prospicio.risk import risk_ratios, statistics
except ImportError as error:
    raise ImportError("this notebook needs a prospicio newer than 0.0.2 (the latest release on PyPI): build it from "
                      "source, `uv run --project python --group examples jupyter lab` from the repository root") from error

N_SIMS = int(os.environ.get("PROSPICIO_EXAMPLE_SIMS", "10000"))
N_ONE_YEAR = min(N_SIMS, 1000)  # one-year simulations for the monitoring section
SAMPLER = dict(chains=4, tune=600, draws=600)  # small on purpose: the demo runs in a few minutes
SEED = 20261023

DATA = next((p / sub for p in [Path.cwd(), *Path.cwd().parents]
             for sub in ("examples/data/submission", "data/submission") if (p / sub).is_dir()), None)
if DATA is None:
    raise FileNotFoundError(
        f"examples/data/submission not found in {Path.cwd()} or above it: open this notebook from inside a clone "
        "of the repository (examples/data/make_submission.py writes the files)")


def read_csv(name):
    with open(DATA / name) as f:
        return list(csv.DictReader(f))


def when(text):
    y, m, d = map(int, text.split("-"))
    return year_fraction(y, m, d)


def money(x):
    return f"{x / 1e6:,.1f}m"


def show(rows, header):
    # A small text table: rows of strings under a header row.
    widths = [max(len(str(r[i])) for r in [header, *rows]) for i in range(len(header))]
    for r in [header, *rows]:
        print("  ".join(str(c).rjust(w) for c, w in zip(r, widths)))


def uniform(i):
    # The i-th point of a low-discrepancy sequence on (0, 1): the same numbers in the R twin, with no generator to differ.
    return (0.5 + i * 0.6180339887498949) % 1.0


def normal(i):
    return NormalDist().inv_cdf(uniform(i))


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs)


def sd(xs):
    xs = list(xs)
    m = mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))


print(f"submission files in {DATA.parent.name}/{DATA.name}; {N_SIMS} simulations, seed {SEED}")

## 1. The submission and the cedant's triangle

A real submission comes as spreadsheets, and here it is CSV files. The analysis needs three items from it:

* the loss triangle (paid and incurred by accident year and age),
* the premium by year,
* the rate changes.

The rate changes restate the premium of each year at the **current rate level** (the *on-level* premium). Loss ratios are comparable across years only on that basis. The 2026 loss ratio forecast is on the 2026 on-level premium.

In [ ]:
program = {r["item"]: r["value"] for r in read_csv("treaty_program.csv")}
SUBJECT_PREMIUM = {int(r["calendar_year"]): float(r["subject_premium"]) for r in read_csv("subject_premium.csv")}
YEARS = sorted(SUBJECT_PREMIUM)
P_2026 = float(program["estimated_subject_premium_income"])

rate_rows = read_csv("rate_changes.csv")
history = RateHistory([when(r["effective_date"]) for r in rate_rows],
                      [float(r["average_rate_change"]) for r in rate_rows])
onlevel = {y: SUBJECT_PREMIUM[y] * history.rate_level(Period.calendar(y, y + 1), 1.0).on_level_factor for y in YEARS}

tri_rows = read_csv("triangle.csv")
triangle = Triangle.from_long(
    [int(r["accident_year"]) for r in tri_rows],
    [int(r["development_months"]) for r in tri_rows],
    {"paid": [float(r["paid"]) for r in tri_rows],
     "incurred": [float(r["incurred"]) for r in tri_rows],
     "onlevel": [onlevel[int(r["accident_year"])] for r in tri_rows]},
)
print(f"cedant {program['cedant']}; {program['line_of_business']}; treaty year {program['period_start']} to {program['period_end']}")
print(f"estimated 2026 subject premium {money(P_2026)}; triangle origins {triangle.origins[0]} to {triangle.origins[-1]}, "
      f"{len(triangle.development)} ages")

## 2. Historical exhibits and a backtest

Two exhibits show the history to the fund before it trusts a model. The **paid chain ladder** gives the cedant's payment pattern (the share of the ultimate loss paid by each age), which the cash-flow step needs. The **loss ratio by accident year** at its latest age, on on-level premium, shows the level and the spread to compare with the prior.

In [ ]:
chain = ChainLadder().fit(triangle, "paid")
CDF = list(chain.cdf)
AGES = [12 * (i + 1) for i in range(len(CDF))]
latest = {int(r["accident_year"]): (float(r["paid"]), float(r["incurred"]))  # last row per year is the latest age
          for r in tri_rows}
loss_ratio_now = {y: latest[y][1] / onlevel[y] for y in YEARS}

print("paid chain ladder: age (months), factor to ultimate, share paid")
show([(a, f"{c:.3f}", f"{1 / c:.1%}") for a, c in zip(AGES, CDF)], ("age", "cdf", "paid"))
print()
show([(y, money(onlevel[y]), money(latest[y][1]), f"{loss_ratio_now[y]:.1%}") for y in YEARS],
     ("year", "on-level prem", "incurred now", "incurred LR"))

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
left.plot([0, *[a / 12 for a in AGES]], [0, *[1 / c for c in CDF]], marker="o")
left.set_xlabel("years since the start of the accident year")
left.set_ylabel("share of ultimate paid")
left.set_title("Paid pattern (chain ladder)")
right.bar([str(y) for y in YEARS], [loss_ratio_now[y] for y in YEARS])
right.set_ylabel("incurred loss ratio at latest age")
right.set_title("Loss ratio by accident year")
right.tick_params(axis="x", rotation=60)
fig.tight_layout()
plt.show()

**How well does the model forecast this cedant?** A **diagonal backtest** holds out the latest calendar diagonals. For each diagonal, it fits the model on the data known before that diagonal and scores the forecast of the cells that came next. Here the model is the Bayesian over-dispersed Poisson chain ladder on two diagonals, with a small sampler to keep the notebook fast. A real analysis would hold out more diagonals and add the other models that the fund considers (`diagonal_backtest` takes several). Coverage is the share of held-out cells inside the central 90% interval. Actual versus expected is the ratio of the actual cells to the expected cells.

In [ ]:
backtest = diagonal_backtest(triangle, "paid", {"bayes_odp": BayesOdp(**SAMPLER)}, 2, n_sims=500, seed=SEED)
scores = backtest.mean()
print("held-out diagonals:", ", ".join(backtest.diagonals))
print(scores.to_string(index=False) if hasattr(scores, "to_string") else scores)
if backtest.warnings:
    print("sampler warnings:", backtest.warnings)

## 3. The industry prior as an explicit input

A cedant with ten years of history still gets value from an outside view of the line. This view is a prior for the ultimate loss ratio, with a stated strength. The deal analyst writes it as a **prior mean `m` and coefficient of variation `cv`**, as inputs with a justification. `prospicio` estimates them from several programs with `hierarchical_credibility` (Bühlmann-Straub with a normal prior on the collective mean).

The peer programs below are **synthetic** (#599, #604). A fixed low-discrepancy sequence draws them, so that the R twin has the same values. There are twelve programs with eight years of loss ratios and premium weights. The method is real. The collective mean and the predictive distribution of the loss ratio of a new program give `m` and `cv`. The prior on the collective mean (`collective_mean=(0.50, 0.10)`) is the analyst's judgment about the line.

In [ ]:
N_PEERS, N_YEARS = 12, 8
peer_mean = [0.48 + 0.06 * normal(i) for i in range(N_PEERS)]
peer_weight = [[0.5 + 1.5 * uniform(100 + i * N_YEARS + j) for j in range(N_YEARS)] for i in range(N_PEERS)]  # premium, 100m
peer_ratio = [[peer_mean[i] + 0.08 / math.sqrt(peer_weight[i][j]) * normal(1000 + i * N_YEARS + j)
               for j in range(N_YEARS)] for i in range(N_PEERS)]

industry = hierarchical_credibility(peer_ratio, peer_weight, collective_mean=(0.50, 0.10), seed=SEED, **SAMPLER)
# The risk premiums of all groups pooled: the distribution of a program's expected loss ratio in this market.
pooled = [x for row in industry.predictive(n_sims=2000, seed=SEED).draw_matrix() for x in row]
M_PRIOR = mean(pooled)
CV_PRIOR = sd(pooled) / M_PRIOR
print(f"collective mean {industry.collective_mean:.3f}; a program's expected loss ratio: mean {M_PRIOR:.3f}, "
      f"sd {sd(pooled):.3f}, cv {CV_PRIOR:.3f}")
print(f"inputs used below: m = {M_PRIOR:.3f}, cv = {CV_PRIOR:.3f}")

## 4. The cedant's ultimates against that prior

`BayesBfNormalised` is the Bayesian Bornhuetter-Ferguson with the prior on the **expected ultimate** of each accident year. `apriori` is the expected ultimate loss ratio, and `cv` sets how firmly the model holds it. A flat prior (`cv=inf`) gives the Bayesian chain ladder, and a very tight prior gives the classical Bornhuetter-Ferguson. The industry prior as `m` and `cv` is the usual compromise. The cedant's data carry the most weight where they are informative (old years), and the prior steadies the immature years.

The model fits the **paid** triangle on on-level premium, so that its pattern is the payment pattern that the cash flows need. The model has no tail beyond the last age.

In [ ]:
def build(rows):
    return Triangle.from_long(
        [int(r["accident_year"]) for r in rows], [int(r["development_months"]) for r in rows],
        {"paid": [float(r["paid"]) for r in rows], "onlevel": [onlevel[int(r["accident_year"])] for r in rows]})



# The oldest age is dropped: only 2016 has it, the chain ladder shows no development there, and the model needs
# every column's increments to sum to more than zero (the one cell at 120 months is 3,000 below the one at 108).
fit_rows = [r for r in tri_rows if int(r["development_months"]) <= 108]
latest_fit = {int(r["accident_year"]): float(r["paid"]) for r in fit_rows}
model = BayesBfNormalised(apriori=M_PRIOR, cv=CV_PRIOR, n_sims=N_SIMS, seed=SEED, **SAMPLER)
fit = model.fit(build(fit_rows), "paid", "onlevel")

origins = [int(o) for o in fit.origins]
exposure = dict(zip(origins, fit.exposure))
ultimate_lr = {}  # per origin: simulated ultimate loss ratio draws (latest paid plus the reserve draws)
for o in origins:
    reserve = fit.reserves.marginal((str(o),)).draws
    ultimate_lr[o] = [(latest_fit[o] + r) / exposure[o] for r in reserve]
post_mean = {o: mean(ultimate_lr[o]) for o in origins}
post_cv = {o: sd(ultimate_lr[o]) / post_mean[o] for o in origins}

print("ultimate loss ratio, paid basis: posterior mean, cv, and the prior mean")
show([(o, f"{post_mean[o]:.3f}", f"{post_cv[o]:.3f}", f"{M_PRIOR:.3f}") for o in origins],
     ("year", "posterior", "cv", "prior"))
print("pattern (share paid at each age):", ", ".join(f"{p:.3f}" for p in fit.pattern))
print("sampler diagnostics:", {k: v for k, v in fit.diagnostics.items() if k in ("max_rhat", "min_ess_bulk", "divergences")})

## 5. The 2026 loss ratio

The model forecasts the cells of the years in the triangle. Section 4 turns its reserve draws into a loss ratio for each simulation: latest paid plus reserve draws, divided by premium (#602). For 2026, a new accident year, the cell below uses a **lognormal** (#598).

The mean of the lognormal is the average of the posterior mean ultimate loss ratios of the last five years. Loss ratios increase over the ten years and the model has no trend, so older years would understate 2026. The coefficient of variation combines three parts:

* the spread of the means of those five years (the variation from year to year),
* the average posterior cv of one year (estimation and process error),
* the cv of the industry prior, as the uncertainty about the level in 2026 (the model has no loss trend).

The lognormal is a substitute for a predictive distribution. The cell prints its mean and cv, so that a reviewer can challenge them.

In [ ]:
RECENT = origins[-5:]  # loss ratios have drifted up over the ten years, so the level is read from the last five
MU_2026 = mean(post_mean[o] for o in RECENT)
CV_BETWEEN = sd([post_mean[o] for o in RECENT]) / MU_2026
CV_WITHIN = math.sqrt(mean(post_cv[o] ** 2 for o in RECENT))
CV_LEVEL = CV_PRIOR  # uncertainty about the level in 2026: the model has no loss trend, so the prior's own cv is added
CV_2026 = math.sqrt(CV_BETWEEN ** 2 + CV_WITHIN ** 2 + CV_LEVEL ** 2)
sigma = math.sqrt(math.log(1 + CV_2026 ** 2))
mu = math.log(MU_2026) - sigma ** 2 / 2
LR = [math.exp(mu + sigma * normal(10_000 + i)) for i in range(N_SIMS)]  # the 2026 ultimate loss ratio, one per simulation
print(f"2026 loss ratio: mean {mean(LR):.3f} (target {MU_2026:.3f}), cv {sd(LR) / mean(LR):.3f} "
      f"(between years {CV_BETWEEN:.3f}, within a year {CV_WITHIN:.3f}, level {CV_LEVEL:.3f})")
print("percentiles 5, 50, 95:", ", ".join(f"{sorted(LR)[int(N_SIMS * p)]:.3f}" for p in (0.05, 0.5, 0.95)))

## 6. The structure

The fund takes 20% of the cedant's subject premium and of its losses: a **quota share**. The reinsurer pays a fixed share of each loss and receives that share of the premium, less a ceding commission to the cedant and brokerage. The notebook compares two commission terms:

* **A, flat**: a 32% ceding commission in all outcomes.
* **B, sliding scale**: the commission decreases as the loss ratio increases. It goes from 36% at a loss ratio of 45% to 24% at 70%, through 32% at 52%. The cedant gets more commission for good results, and the fund shares less of the bad results.

`Tower.apply_annual` applies the contract to the 2026 annual losses (the loss ratio times the premium), one for each simulation. `deal_ratios` gives the loss ratio, the commission, the brokerage and the margin of each simulation. The ceded premium is `0.2 * P`.

In [ ]:
CESSION, BROKERAGE = 0.20, 0.025
CEDED_PREMIUM = CESSION * P_2026
annual_loss = [lr * P_2026 for lr in LR]  # the cedant's whole-book loss, one per simulation

flat = (Layer.quota_share("A_flat", CESSION).with_premium_rate(1.0, P_2026)
        .with_ceding_commission(0.32).with_brokerage(BROKERAGE))
sliding = (Layer.quota_share("B_sliding", CESSION).with_premium_rate(1.0, P_2026)
           .with_sliding_scale([(0.36, 0.45), (0.32, 0.52), (0.24, 0.70)]).with_brokerage(BROKERAGE))

deals = {}
for layer in (flat, sliding):
    tower = Tower([layer])
    result = tower.apply_annual(annual_loss)
    ratios = tower.deal_ratios(result, layer.name)
    column = lambda name: ratios.marginal([name]).draws
    deals[layer.name] = dict(tower=tower, result=result,
                             loss=column("loss_ratio"),
                             commission=[c + b for c, b in zip(column("ceding_commission"), column("brokerage"))],
                             margin=column("margin"))
rows = [(name, f"{mean(d['loss']):.3f}", f"{mean(d['commission']):.3f}", f"{mean(d['margin']):+.3f}",
         f"{sd(d['margin']):.3f}") for name, d in deals.items()]
show(rows, ("structure", "mean LR", "mean comm+brkg", "mean margin", "sd margin"))
print(f"ceded premium {money(CEDED_PREMIUM)}")

## 7. Collateral and cash flows

The money moves monthly. The parts:

* **Payment pattern.** The factors of the paid chain ladder give the share of the ultimate loss paid by each age (`PaymentPattern.from_development`). `CashflowTerms` takes cumulative *monthly* shares, so the cell below interpolates the annual pattern linearly to months (#601). All simulations use the same pattern (#600).
* **Premium.** The cedant writes premium evenly over 2026 and collects it as written. The premium earns over the following two years, because a year of policies written evenly earns over 24 months.
* **Collateral.** The trust must hold 30% of the unearned premium plus 125% of the outstanding loss (the ultimate loss of the deal less paid to date). This rule is a `CollateralRequirement`. The fund posts the shortfall of the trust when it occurs and takes back the excess over the requirement. Thus most of the capital goes in at the start and comes back as the trust pays losses.
* **Yield, expenses and commutation.** The trust earns 3% a year, and expenses are 0.5% a year of the requirement. At month 60 the deal commutes: the trust pays the cedant the outstanding loss, and the fund takes the remainder.

The cash-flow engine takes the ultimate loss ratio of the **deal** (ceded loss over ceded premium) in each simulation and returns the investor's flows. The premium into the trust is `collected_rate` times the premium, a fixed value in the terms. The sliding scale makes the commission different in each simulation. Thus the cell groups the simulations by commission rate, to 0.1 percentage point. It runs the engine once for each group with the `collected_rate` of that group, and puts the flows back in simulation order.

In [ ]:
T = 60  # commutation at the end of month 59
pattern = PaymentPattern.from_development([a / 12 for a in AGES], [*[max(1.0, c) for c in CDF[:-1]], 1.0])


def paid_share(month):
    # Cumulative share of the ultimate paid by the end of `month`: the annual pattern, interpolated linearly.
    t = month / 12
    lo = int(t)
    a = 1 - pattern.outstanding(lo) if lo > 0 else 0.0
    b = 1 - pattern.outstanding(lo + 1)
    return a + (b - a) * (t - lo)


def earned_share(month):
    # Policies written evenly over a year, each earning over twelve months.
    t = month / 12
    return t * t / 2 if t <= 1 else min(1.0, 1 - (2 - t) ** 2 / 2)


written = [min(1.0, (m + 1) / 12) for m in range(T)]
earned = [earned_share(m + 1) for m in range(T)]
paid = [paid_share(m + 1) for m in range(T)]
print(f"payment pattern: mean time to payment {pattern.duration:.2f} years; paid by month 12, 24, 60: "
      f"{paid[11]:.1%}, {paid[23]:.1%}, {paid[59]:.1%}")

requirement = CollateralRequirement({"unearned-premium": 0.30, "outstanding": 1.25})


def make_terms(commission):
    return CashflowTerms(CEDED_PREMIUM, T, written, earned, paid, requirement,
                         commutation_requirement=CollateralRequirement({"outstanding": 1.0}),
                         collected_rate=1 - commission, annual_yield=0.03, expense_rate=0.005)


def run_structure(u, commission):
    # One engine run for each distinct commission rate, flows back in simulation order.
    groups = {}
    for i, c in enumerate(commission):
        groups.setdefault(round(c, 3), []).append(i)
    flows, funded, opening = [None] * len(u), [0.0] * len(u), [0.0] * len(u)
    for c, index in groups.items():
        out = make_terms(c).run([u[i] for i in index])
        out_flows, out_funded, out_opening = out.flows, out.funded, out.opening  # each access copies the whole list
        for k, i in enumerate(index):
            flows[i], funded[i], opening[i] = out_flows[k], out_funded[k], out_opening[k]
    return flows, funded, opening


for name, d in deals.items():
    flows, funded, opening = run_structure(d["loss"], d["commission"])
    d["flows"], d["funded"], d["opening"] = flows, funded, opening
    d["run"] = (DealRun.from_tower(name, d["tower"], d["result"], name)
                .with_flows(flows, funded, opening).with_patterns(written, earned, paid))
    # A book of one deal: its IRR and MOIC in each simulation. The IRR is searched up to 500% a year: the default
    # upper bound, 100%, is below the best years of this deal, which would then show no root and drop out.
    book_cashflows = DealBook([d["run"]]).cashflows(upper=5.0)
    d["rate"], d["moic"] = list(book_cashflows.irr.rate), list(book_cashflows.moic)
print(f"{len(deals)} structures run, {N_SIMS} simulations each, {T} months")
print("capital called from the fund over the term, summed (mean; it is recycled as losses are paid):", ", ".join(f"{n} {money(mean(d['funded']))}" for n, d in deals.items()))

## 8. What the investor gets

For each structure, the cell computes the return of each simulation:

* the **IRR**, the annual internal rate of return of the monthly flows. The cell searches it between -95% and 500% a year. The default search stops at 100%, below the IRR of this deal's best years, which would then show no root and drop out of the percentiles, biasing them down (#670). The cell still excludes and counts any simulation with no root in the range.
* the **MOIC** (multiple on invested capital): one plus the net flows divided by the capital called from the fund.
* the downside: the probability of a negative IRR or a MOIC below 1, a **TVaR** (the mean of the worst tail) of the return, and risk ratios on the deal margin.

Simulations with a total loss have strongly negative IRRs, so the mean IRR is a poor summary. Use the percentiles.

In [ ]:
def finite(xs):
    return [x for x in xs if not math.isnan(x)]


rows = []
for name, d in deals.items():
    rate = finite(d["rate"])
    ordered = sorted(rate)
    pct = lambda p: ordered[min(len(ordered) - 1, int(p * len(ordered)))]
    worst = ordered[: max(1, len(ordered) // 20)]  # the worst 5% of IRRs
    rows.append((name, f"{mean(d['moic']):.3f}", f"{pct(0.05):+.1%}", f"{pct(0.5):+.1%}", f"{pct(0.95):+.1%}",
                 f"{mean(r < 0 for r in rate):.1%}", f"{mean(m < 1 for m in d['moic']):.1%}", f"{mean(worst):+.1%}",
                 f"{len(d['rate']) - len(rate)}"))
show(rows, ("structure", "mean MOIC", "IRR p5", "IRR p50", "IRR p95", "P(IRR<0)", "P(MOIC<1)", "IRR TVaR5", "no root"))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
for name, d in deals.items():
    left.hist(finite(d["rate"]), bins=60, range=(-0.5, 1.0), alpha=0.55, label=name)
    right.hist(d["moic"], bins=60, range=(0.5, 1.6), alpha=0.55, label=name)
left.set_xlabel("IRR (annual)")
right.set_xlabel("MOIC")
left.set_title("Investor IRR")
right.set_title("Investor MOIC")
left.legend()
fig.tight_layout()
plt.show()

`risk_ratios` takes the margin of each simulation, where large is good. It returns Sharpe, Sortino, Omega and the TVaR ratio about a threshold (here a target margin of 10%), under the conventions in its documentation. The quantity here is the deal's **margin** (one less the combined ratio), not the fund's return. The fund's return needs a capital base, which the deal-level ratios do not have. The IRR and MOIC above give the fund's view.

In [ ]:
for name, d in deals.items():
    r = risk_ratios(d["margin"], threshold=0.10, tvar_levels=[95.0])
    print(name, {k: round(v, 3) for k, v in r.items()})

## 9. Comparing the structures

`DealBook.summary()` shows the premium, loss and deductions of the structures side by side, per simulation. A `DealBook` adds its deals on the same simulated years, so its **total** row would be a portfolio that holds both structures. Read only the rows of each structure. The comparison is the purpose of the exercise. The sliding scale gives the fund a better result when losses are high, and less when losses are low.

In [ ]:
book = DealBook([d["run"] for d in deals.values()])
summary = book.summary()
rows = [(summary["deal"][i], summary["item"][i], f"{summary['mean'][i] / 1e6:,.2f}m", f"{summary['ratio'][i]:.3f}")
        for i in range(len(summary["deal"])) if summary["deal"][i] != "total"]
show(rows, ("deal", "item", "mean", "ratio to premium"))

## 10. Monitoring after binding

After binding, each quarter brings a new diagonal of actual paid losses. The fund asks: **was the latest year within the expected range?** The check below holds out the latest diagonal (the 2025 valuation) and fits the model on the data known one year earlier. It takes the **one-year view** (the distribution of the claims development result over the next year) and compares it with the actual diagonal.

The realised result is the opening reserve, less the amount paid in the year, less the closing reserve, for each accident year in the triangle. The model fitted to the full triangle estimates the closing reserve. A realised total in the middle of the forecast shows that the model tracks the data. A total in the tail is a signal to investigate.

This check is on the cedant's *history* (#603).

In [ ]:
def valuation_year(row):
    return int(row["accident_year"]) + int(row["development_months"]) // 12 - 1


last = max(valuation_year(r) for r in fit_rows)
opening_rows = [r for r in fit_rows if valuation_year(r) < last]
tri_open = build(opening_rows)
paid_open = {}
for r in opening_rows:
    paid_open[int(r["accident_year"])] = float(r["paid"])

model_open = BayesBfNormalised(apriori=M_PRIOR, cv=CV_PRIOR, n_sims=N_ONE_YEAR, seed=SEED, **SAMPLER)
view = model_open.one_year(tri_open, "paid", "onlevel", n_sims=N_ONE_YEAR)
cdr = view.cdr.total()

opening_reserve = dict(zip((int(o) for o in view.origins), view.opening_reserve))
closing_reserve = {o: mean(fit.reserves.marginal((str(o),)).draws) for o in origins}
paid_in_year = {o: latest_fit[o] - paid_open[o] for o in opening_reserve}
realised = sum(opening_reserve[o] - paid_in_year[o] - closing_reserve[o] for o in opening_reserve)
percentile = mean(x <= realised for x in cdr.draws)
print(f"valuation {last}: one-year claims development result, forecast mean {money(cdr.mean())}, "
      f"sd {money(sd(cdr.draws))}; realised {money(realised)}, at the {percentile:.0%} percentile of the forecast")

## 11. A summary for the investor

One table of the results, as a plain printout of the quantities above (#605). A report for an investment committee would add the contract terms, the data vintage and the diagnostics of the model.

In [ ]:
report = [
    ("2026 loss ratio, mean (cv)", f"{mean(LR):.1%} ({sd(LR) / mean(LR):.0%})"),
    ("industry prior m (cv)", f"{M_PRIOR:.1%} ({CV_PRIOR:.0%})"),
    ("ceded premium", money(CEDED_PREMIUM)),
    ("payment pattern, mean time to payment", f"{pattern.duration:.2f} years"),
    ("backtest sampler warnings", str(len(backtest.warnings))),
    ("monitoring: realised CDR percentile", f"{percentile:.0%}"),
]
for name, d in deals.items():
    rate = finite(d["rate"])
    report += [(f"{name}: median IRR / mean MOIC", f"{sorted(rate)[len(rate) // 2]:+.1%} / {mean(d['moic']):.3f}"),
               (f"{name}: P(IRR < 0)", f"{mean(r < 0 for r in rate):.1%}")]
show([(k, v) for k, v in report], ("item", "value"))

## Issues for the workarounds

| Ref | Step in this notebook | Issue |
|---|---|---|
| G1 | the lognormal for 2026 in section 5 | #598 |
| G2 | the synthetic peers in section 3 | #599 |
| G3 | one payment pattern for all simulations in section 7 | #600 |
| G4 | the monthly interpolation of the pattern in section 7 | #601 |
| G5 | the loop over `reserves` in section 4 | #602 |
| G6 | the monitoring in section 10 | #603 |
| G7 | the prior inputs in section 3 | #604 |
| G8 | the printout in sections 1 and 11 | #605 |